# Isla-SNN: Dataset Caching

Pre-tokenize and pack a multi-source dataset for training.
Each source is processed one at a time to keep RAM under control.

In [1]:
!pip install -q datasets transformers

## 1. Config

In [ ]:
from pathlib import Path

SEQ_LEN       = 2048
TARGET_TOKENS = 1_000_000_000
FLUSH_EVERY   = 20_000
SEED          = 42

TOKENIZER_NAME = "Polygl0t/Tucano2-qwen-3.7B-Base"

OUTPUT_DIR = Path("./data/isla_1b_tokenized")
CHUNKS_DIR = OUTPUT_DIR / "_chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = [
    {"name": "en",   "path": "codelion/fineweb-edu-1B",      "config": None, "split": "train", "pct": 0.60},
    {"name": "pt",   "path": "PleIAs/Portuguese-PD",         "config": None, "split": "train", "pct": 0.30},
    {"name": "code", "path": "luisroque/instruct-python-500k", "config": None, "split": "train", "pct": 0.05},
    {"name": "math", "path": "AI-MO/NuminaMath-CoT",         "config": None, "split": "train", "pct": 0.05},
]

for s in SOURCES:
    s["budget"] = int(TARGET_TOKENS * s["pct"])
    print(f"  {s['name']:>5}: {s['budget']:>13,} tokens")
print(f"  total: {TARGET_TOKENS:,} tokens")

## 2. Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
EOS = tokenizer.eos_token_id
print(f"Vocab: {len(tokenizer):,} | EOS: {EOS}")

## 3. Process Sources

In [ ]:
import time, gc
from datasets import load_dataset, Dataset, Features, Value

FEAT = Features({"input_ids": [Value("int32")], "labels": [Value("int32")]})

def get_text(ex):
    q = ex.get("question")
    a = ex.get("answer")
    if isinstance(q, str) and isinstance(a, str) and q and a:
        return q + "\n" + a
    for k in ["text", "content", "body", "solution"]:
        v = ex.get(k)
        if isinstance(v, str) and len(v) > 50:
            return v
    return ""

def save_chunk(rows, idx):
    ds = Dataset.from_dict(rows, features=FEAT)
    ds.save_to_disk(str(CHUNKS_DIR / f"c{idx:04d}"))
    n = len(ds)
    del ds; gc.collect()
    return n

def process(src, ci):
    name, budget = src["name"], src["budget"]
    print(f"\n--- {name.upper()} | budget {budget:,} tokens ---")
    kwargs = dict(split=src["split"], streaming=True)
    if src["config"]:
        kwargs["name"] = src["config"]
    try:
        stream = load_dataset(src["path"], **kwargs)
    except Exception:
        if src["path"] == "PleIAs/Portuguese-PD":
            ft = Features({"identifier": Value("string"), "creator": Value("string"), "title": Value("string"), "publication_date": Value("int64"), "word_count": Value("int64"), "text": Value("string")})
            stream = load_dataset(src["path"], split=src["split"], streaming=True, features=ft)
        else:
            raise
    buf, rows, done, t0 = [], {"input_ids": [], "labels": []}, 0, time.time()
    for ex in stream:
        txt = get_text(ex)
        if not txt: continue
        for i in range(0, len(txt), 100_000):
            buf.extend(tokenizer.encode(txt[i:i+100_000], add_special_tokens=False))
        buf.append(EOS)
        while len(buf) >= SEQ_LEN:
            seq = buf[:SEQ_LEN]; buf = buf[SEQ_LEN:]
            rows["input_ids"].append(seq); rows["labels"].append(seq)
            done += SEQ_LEN
            if len(rows["input_ids"]) >= FLUSH_EVERY:
                n = save_chunk(rows, ci)
                spd = done / max(time.time()-t0, 1) / 1e6
                print(f"  c{ci:04d}: {n:,} rows | {done:,}/{budget:,} ({done/budget*100:.1f}%) | {spd:.1f}M t/s")
                ci += 1; rows = {"input_ids": [], "labels": []}
        if done >= budget: break
    if rows["input_ids"]: save_chunk(rows, ci); ci += 1
    print(f"  {name}: {done:,} tokens in {(time.time()-t0)/60:.1f}min")
    del buf, rows, stream; gc.collect()
    return done, ci

chunk_i, grand_total = 0, 0
for src in SOURCES:
    produced, chunk_i = process(src, chunk_i)
    grand_total += produced
print(f"\nDONE: {grand_total:,} tokens in {chunk_i} chunks")

## 4. Merge and Save

In [ ]:
from datasets import load_from_disk, concatenate_datasets, DatasetDict
import gc

paths = sorted(str(p) for p in CHUNKS_DIR.glob("c*"))
print(f"{len(paths)} chunks to merge")

cur = paths
r = 0
while len(cur) > 1:
    nxt = []
    for i in range(0, len(cur), 2):
        if i+1 >= len(cur): nxt.append(cur[i]); continue
        a = load_from_disk(cur[i]); b = load_from_disk(cur[i+1])
        m = concatenate_datasets([a, b])
        out = str(CHUNKS_DIR / f"m{r}_{i//2:03d}")
        m.save_to_disk(out); nxt.append(out)
        del a, b, m; gc.collect()
    cur = nxt; r += 1
    print(f"  round {r}: {len(cur)} parts left")

full = load_from_disk(cur[0])
val_n = max(1, int(len(full) * 0.001))
sp = full.train_test_split(test_size=val_n, seed=SEED, shuffle=True)
del full; gc.collect()

ds = DatasetDict({"train": sp["train"], "validation": sp["test"]})
ds.save_to_disk(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Train: {len(ds['train']):,} | Val: {len(ds['validation']):,}")
print(f"Saved: {OUTPUT_DIR.resolve()}")

## 5. Quick Check

In [ ]:
from datasets import load_from_disk
ds = load_from_disk(str(OUTPUT_DIR))
print(ds)
print(f"Seq len: {len(ds['train'][0]['input_ids'])}")
print(tokenizer.decode(ds['train'][0]['input_ids'][:80]))

# 6. Upload to Huggingface

In [ ]:
from huggingface_hub import login, HfApi
login() 
api = HfApi()
api.create_repo("Heitorkk2/Isla_1b_tokenized", repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(
    folder_path=str(OUTPUT_DIR),
    repo_id="<your_name>/Isla_1b_tokenized",
    repo_type="dataset",
)
print("Done!")